In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["MKL_SERVICE_FORCE_INTEL"]="1"
# os.environ["OMP_NUM_THREADS"] = "8"  # 控制MKL使用的线程数
# os.environ["MKL_NUM_THREADS"] = "8"
# os.environ["BLAS"] = "MKL"
# os.environ["USE_CUSOLVER"] = "1"

import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoFeatureExtractor, SwinForImageClassification,
    ViTImageProcessor, ViTForImageClassification,
    Trainer, TrainingArguments, EvalPrediction,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

from meft import MeftConfig, MeftTrainer
import meft

/home/shijx/miniconda3/envs/loract/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
import numpy as np
from transformers import set_seed as hf_set_seed
seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
hf_set_seed(seed)  # transformers 内部用到的随机也统一
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    num_labels=100,
    ignore_mismatched_sizes=True,
    # device_map="auto",
    device_map="cuda:0",
)

dataset = load_dataset("cifar100")

def transform(examples):
    inputs = processor(
        images=[x.convert("RGB") for x in examples["img"]],
        return_tensors="pt"
    )
    return {
        "pixel_values": inputs.pixel_values,
        "label": examples["fine_label"]
    }

dataset = dataset.with_transform(transform)

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 371,812 || all params: 86,247,368 || trainable%: 0.4311


In [5]:
def compute_metrics(eval_pred: EvalPrediction):
    evaluation = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return evaluation.compute(predictions=predictions, references=labels)

In [6]:
lq_trainer = MeftTrainer[Trainer](
    model=model,
    args=TrainingArguments(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        learning_rate=1e-3,
        weight_decay=1e-2,
        warmup_ratio=0.1,    # 新加
        lr_scheduler_type="cosine",
        bf16=True,
        bf16_full_eval=True,
        # deepspeed={
        #     "train_batch_size": "auto",
        #     "gradient_accumulation_steps": "auto",
        #     "zero_optimization": {
        #         "stage": 1,
        #     },
        # },
        use_liger_kernel=True,
        logging_steps=10,
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
    ),
    data_collator=None,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    meft_config=MeftConfig(
        patch_locations=(
            "norm",
            # "attn_in",
            # "attn_out",
            # "mlp_in",
            # "mlp_out",
            # "act_fn",
            "ckpt_attn",
            "ckpt_mlp",
            # "ckpt_layer",
        ),
        compress_kwargs={
            "rank": 128,
            # "niter": 1,
            "lowrank_plus_quantization": True,
        },
        # compress_workers=2,
    ),
)

Applying patch to vit model in: ('norm', 'ckpt_attn', 'ckpt_mlp')


/home/shijx/chenxy/LoRAct/meft/patch/models/vit.py:72: CheckpointViTMLPWarning: ViT only supports checkpointing the first layer of MLP.
  warnings.warn("ViT only supports checkpointing the first layer of MLP.", CheckpointViTMLPWarning)


In [7]:
lq_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,4.600400
20,4.522900
30,4.375400
40,4.074600
50,3.570500
60,2.996100
70,2.425500
80,1.921300
90,1.539800
100,1.237000


TrainOutput(global_step=391, training_loss=1.2043697698341915, metrics={'train_runtime': 394.2153, 'train_samples_per_second': 126.834, 'train_steps_per_second': 0.992, 'total_flos': 3.8947931430912e+18, 'train_loss': 1.2043697698341915, 'epoch': 1.0})

In [8]:
lq_trainer.evaluate()

{'eval_loss': 0.462003231048584,
 'eval_accuracy': 0.9028,
 'eval_runtime': 58.5549,
 'eval_samples_per_second': 170.78,
 'eval_steps_per_second': 5.345,
 'epoch': 1.0}